# Stage 04 — Data Acquisition and Ingestion (Lecture)

**Goals:** API pull, scraping, secrets via `.env`, validation, saving to `data/raw/`.

> Ethics & legality: obey site Terms, robots.txt, and rate limits. Do not scrape where prohibited.

In [1]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
!pip install pandas
!pip install requests
!pip install yfinance
!pip install python-dotenv
!pip install beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 35.5 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: cffi
    Found existing installation: cffi 1.17.1
    Uninstalling cffi-1.17.1:
      Successfully uninstalled cffi-1.17.1
  Attempting uninstall: certifi
    Found existing installation: certifi 2022.12.7
    Uninstalling certifi-2022.12.7:
      Successfully uninstalled certifi-2022.12.7


In [2]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    (".env", "NEEDED", "YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing"),
    (".env.example", "NEEDED", "shipped with this stage - the template you copy to .env"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: /Users/joycelu/Desktop/NYU/Bootcamp/lecture

  [OK ]  NEEDED    .env                                YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing
  [OK ]  NEEDED    .env.example                        shipped with this stage - the template you copy to .env

All needed files present.


In [4]:
import os, json, time, datetime as dt, csv, pathlib
from typing import Dict, List
import requests
import pandas as pd
#bs4 是 Python 的网页解析包 Beautiful Soup 4，主要用于从 HTML 或 XML 中提取数据，经常用于网络爬虫
from bs4 import BeautifulSoup
from dotenv import load_dotenv

DATA_RAW = pathlib.Path("data/raw")
#创建文件夹，自动放在当前.cwd文件夹
DATA_RAW.mkdir(parents=True, exist_ok=True)#表示如果data文件夹不存在，先创建data/，再创建raw/

load_dotenv()

# No key yet? A free one takes about 30 seconds:
#     https://www.alphavantage.co/support/#api-key
# Put it in a .env file beside this notebook:
#     ALPHAVANTAGE_API_KEY=your_key_here
# Never hard-code it in the notebook - notebooks get shared, committed and screen-shared.
# Without a key this notebook still runs; it falls back to yfinance below.
ALPHA_KEY = os.getenv("ALPHAVANTAGE_API_KEY")
print("Loaded ALPHAVANTAGE_API_KEY?", bool(ALPHA_KEY))

Loaded ALPHAVANTAGE_API_KEY? True


## Helper functions: validation & filenames

In [5]:
def safe_stamp():
    return dt.datetime.now().strftime("%Y%m%d-%H%M%S")
safe_stamp

<function __main__.safe_stamp()>

In [6]:
def safe_filename(prefix: str, meta: Dict[str, str]) -> str:
    mid = "_".join([f"{k}-{str(v).replace(' ', '-')[:20]}" for k, v in meta.items()])
    return f"{prefix}_{mid}_{safe_stamp()}.csv"

In [7]:
safe_filename("wiki",{"param1":1,"param2":2})

'wiki_param1-1_param2-2_20260814-103518.csv'

### Python aside — comprehensions, and what type hints do *not* do

A short detour before the ingestion code, through the Python it relies on: building a
list with a loop versus a comprehension, filtering and conditional expressions inside
one, and the fact that type annotations are **not enforced at runtime** — the cell
below passes a dict to a parameter annotated `str`, and it runs happily.

We return to data acquisition at *API Ingestion* below.


In [8]:
numbered_list = []
for idx,i in enumerate(["temperature","humidity"]):
    numbered_list += [str(idx+1)+"."+str(i)]
numbered_list

[str(idx+1)+"."+str(i)  for idx,i in enumerate(["temperature","humidity"])]

# [i[:2]  for i in ["temperature","humidity"]]

['1.temperature', '2.humidity']

In [9]:
def test(x:str):
    return(x)
test({"1":1})

{'1': 1}

In [10]:
x = 5
True     if x == 5    else   False
[ c*2 for c in [1,2,3,4] ]
[ c*2 for c in [1,2,3,4] if c !=3]

[2, 4, 8]

In [11]:
[None if c not in [1,2] else c for c in [1,2,3,4] ]

[1, 2, None, None]

In [13]:
def validate_df(df: pd.DataFrame, required_cols: List[str], dtypes_map: Dict[str, str]) -> Dict[str, str]:
    msgs = {}
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        msgs['missing_cols'] = f"Missing columns: {missing}"
    for col, dtype in dtypes_map.items():
        if col in df.columns:
            try:
                if dtype == 'datetime64[ns]':
                    pd.to_datetime(df[col])
                elif dtype == 'float':
                    pd.to_numeric(df[col])
            except Exception as e:
                msgs[f'dtype_{col}'] = f"Failed to coerce {col} to {dtype}: {e}"
    na_counts = df.isna().sum().sum()
    msgs['na_total'] = f"Total NA values: {na_counts}"
    return msgs

## API Ingestion: Alpha Vantage (fallback: yfinance)
- If `ALPHAVANTAGE_API_KEY` is set, use Alpha Vantage `TIME_SERIES_DAILY`.
- Else, demonstrate with `yfinance` (no key).
- **Adjusted close is a paid field at Alpha Vantage**, so both branches take the raw `close`. Worth pausing on: a vendor moved a column behind a paywall, and any pipeline that assumed the column existed broke. Code defensively against the *shape* of what comes back, not the shape you remember.
- The free tier allows **25 calls a day**. Past that the API still answers `200 OK` with an explanation instead of data - which is why the code checks for the series rather than trusting the status code.

In [14]:
SYMBOL = "AAPL"

use_alpha = bool(ALPHA_KEY)
print("Using Alpha Vantage:", use_alpha)

if use_alpha:
    url = "https://www.alphavantage.co/query"
    params = {
        "function": "TIME_SERIES_DAILY",
        "symbol": SYMBOL,
        "outputsize": "compact",
        "apikey": ALPHA_KEY,
        "datatype": "json"
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()
    key = [k for k in js.keys() if "Time Series" in k]
    if not key:
        # Alpha Vantage replies HTTP 200 with a prose blob - not an error status - when
        # the free tier's daily cap is hit or the endpoint has moved to premium, so
        # raise_for_status() above sees nothing wrong. Say so and fall back.
        print("Alpha Vantage returned no series:", str(list(js.values())[0])[:150])
        use_alpha = False

if use_alpha:
    series = js[key[0]]
    df_api = (pd.DataFrame(series).T
              .rename_axis('date')
              .reset_index())
    # keep a couple columns and coerce types
    df_api = df_api[['date', '4. close']].rename(columns={'4. close': 'close'})
    df_api['date'] = pd.to_datetime(df_api['date'])
    df_api['close'] = pd.to_numeric(df_api['close'])

if not use_alpha:
    import yfinance as yf
    df_api = yf.download(SYMBOL, period="6mo", interval="1d", auto_adjust=False,
                          multi_level_index=False).reset_index()[['Date','Close']]
    df_api.columns = ['date','close']

df_api = df_api.sort_values('date').reset_index(drop=True)
msgs = validate_df(df_api, required_cols=['date','close'], dtypes_map={'date':'datetime64[ns]','close':'float'})
print(msgs)

fname = safe_filename(prefix="api", meta={"source": "alpha" if use_alpha else "yfinance", "symbol": SYMBOL})
out_path = DATA_RAW / fname
df_api.to_csv(out_path, index=False)
print("Saved:", out_path)

Using Alpha Vantage: True
{'na_total': 'Total NA values: 0'}
Saved: data/raw/api_source-alpha_symbol-AAPL_20260814-110708.csv


## Scraping a Simple Public Table with BeautifulSoup
*Use responsibly. Provide a polite User-Agent and delays if looping.*

In [15]:
SCRAPE_URL = "https://en.wikipedia.org/wiki/Dow_Jones_Industrial_Average"
headers = {"User-Agent": "AFE-Course-Notebook/1.0 (contact: instructor@example.edu)"}
try:
    resp = requests.get(SCRAPE_URL, headers=headers, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    table = soup.find('table', class_='wikitable')   # the components table
    rows = []
    for tr in table.find_all('tr'):
        cells = [td.get_text(strip=True) for td in tr.find_all(['td','th'])]
        if cells:
            rows.append(cells)
    # assume first row is header
    header, *data = rows
    df_scrape = pd.DataFrame(data, columns=header)
except Exception as e:
    print("Scrape failed (demoing with inline HTML).", e)
    html = """
    <table>
      <tr><th>Ticker</th><th>Price</th></tr>
      <tr><td>AAA</td><td>101.2</td></tr>
      <tr><td>BBB</td><td>98.7</td></tr>
    </table>
    """
    soup = BeautifulSoup(html, 'html.parser')
    rows = []
    for tr in soup.find_all('tr'):
        cells = [td.get_text(strip=True) for td in tr.find_all(['td','th'])]
        if cells:
            rows.append(cells)
    header, *data = rows
    df_scrape = pd.DataFrame(data, columns=header)

if 'Price' in df_scrape.columns:
    df_scrape['Price'] = pd.to_numeric(df_scrape['Price'], errors='coerce')

msgs2 = validate_df(df_scrape, required_cols=list(df_scrape.columns), dtypes_map={})
print(msgs2)

fname2 = safe_filename(prefix="scrape", meta={"site": "wikipedia", "table": "djia"})
out_path2 = DATA_RAW / fname2
df_scrape.to_csv(out_path2, index=False)
print("Saved:", out_path2)

{'na_total': 'Total NA values: 0'}
Saved: data/raw/scrape_site-wikipedia_table-djia_20260814-112416.csv


## Notes & Sources
- API Source: Alpha Vantage or yfinance fallback
- Scrape Source: replace `SCRAPE_URL` with permitted page
- Secrets: `.env` with `ALPHAVANTAGE_API_KEY`; do not commit `.env`

### Appendix - Previously working examples

## Scraping S&P 500 Constituents from Wikipedia (dynamic validation)

In [ ]:
SCRAPE_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {"User-Agent": "AFE-Course-Notebook/1.0 (contact: instructor@example.edu)"}

try:
    resp = requests.get(SCRAPE_URL, headers=headers, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    table = soup.find('table', id='constituents')
    if table is None:
        raise RuntimeError("Could not find table with id='constituents'")
    rows = []
    for tr in table.find_all('tr'):
        cells = [td.get_text(strip=True) for td in tr.find_all(['td','th'])]
        if cells:
            rows.append(cells)
    header, *data = rows
    df_scrape = pd.DataFrame(data, columns=header)
    for col in df_scrape.columns:
        if col.lower() in ['cik']:
            df_scrape[col] = pd.to_numeric(df_scrape[col], errors='coerce')
        elif 'date' in col.lower():
            df_scrape[col] = pd.to_datetime(df_scrape[col], errors='coerce')
except Exception as e:
    print("Scrape failed (demoing with inline HTML).", e)
    html = """
    <table>
      <tr><th>Ticker</th><th>Price</th></tr>
      <tr><td>AAA</td><td>101.2</td></tr>
      <tr><td>BBB</td><td>98.7</td></tr>
    </table>
    """
    soup = BeautifulSoup(html, 'html.parser')
    rows = []
    for tr in soup.find_all('tr'):
        cells = [td.get_text(strip=True) for td in tr.find_all(['td','th'])]
        if cells:
            rows.append(cells)
    header, *data = rows
    df_scrape = pd.DataFrame(data, columns=header)

dtypes_map = {}
for col in df_scrape.columns:
    if pd.api.types.is_numeric_dtype(df_scrape[col]):
        dtypes_map[col] = 'float'
    elif pd.api.types.is_datetime64_any_dtype(df_scrape[col]):
        dtypes_map[col] = 'datetime64[ns]'

msgs2 = validate_df(df_scrape, required_cols=list(df_scrape.columns), dtypes_map=dtypes_map)
print(msgs2)

fname2 = safe_filename(prefix="scrape", meta={"site": "wikipedia", "table": "sp500"})
out_path2 = DATA_RAW / fname2
df_scrape.to_csv(out_path2, index=False)
print("Saved:", out_path2)

## Optional Example — Pulling & Parsing a Non-Structured HTML Page

`example.com` is a deliberately MINIMAL page - 559 bytes, 11 tags in total. That makes it a useful baseline: this is what parsing looks like when the HTML is clean and tiny. Compare it with the BBC page below, which is ~377 KB and 600+ `<div>`s.

This example demonstrates:
- Downloading and saving a raw page
- Saving it to disk
- Reloading and inspecting HTML snippets
- Counting tags and summarizing structure
- Extracting some “unexpected” data from arbitrary content

Worth emphasising: the two pages together are the lesson. HTML is written for browsers, not for you — `example.com` is the friendliest case you will ever get, and the BBC page is the honest one. Scraping is brittle by nature, so treat any selector you write as something that will break when the site changes.

In [16]:
import collections

URL_MISC = "https://www.example.com"  # benign demo site
html_path = DATA_RAW / f"example_site_{safe_stamp()}.html"

resp = requests.get(URL_MISC, headers={"User-Agent": "AFE-Course-Notebook/1.0"}, timeout=30)
resp.raise_for_status()
with open(html_path, "w", encoding="utf-8") as f:
    f.write(resp.text)
print(f"Downloaded and saved raw HTML to: {html_path}")

with open(html_path, "r", encoding="utf-8") as f:
    html_text = f.read()
print("\n=== First 500 chars of HTML ===")
print(html_text[:500])

soup_misc = BeautifulSoup(html_text, "html.parser")

print("\n=== First H1 tags ===")
for tag in soup_misc.find_all("h1")[:3]:
    print(tag.get_text(strip=True))

print("\n=== First H2 tags ===")
for tag in soup_misc.find_all("h2")[:3]:
    print(tag.get_text(strip=True))

print("\n=== First P tags ===")
for tag in soup_misc.find_all("p")[:3]:
    print(tag.get_text(strip=True))

tag_counts = collections.Counter([tag.name for tag in soup_misc.find_all(True)])
print("\n=== Tag frequency summary ===")
for tag, count in tag_counts.most_common(10):
    print(f"{tag}: {count}")

links = [a['href'] for a in soup_misc.find_all("a", href=True) if a['href'].startswith("http")]
print("\n=== External Links (first 5) ===")
for link in links[:5]:
    print(link)

meta_info = [(m.get("name"), m.get("content")) for m in soup_misc.find_all("meta") if m.get("name")]
print("\n=== Meta tags found ===")
for name, content in meta_info:
    print(f"{name}: {content}")

Downloaded and saved raw HTML to: data/raw/example_site_20260814-112817.html

=== First 500 chars of HTML ===
<!doctype html><html lang="en"><head><title>Example Domain</title><link rel="icon" href="data:,"><meta name="viewport" content="width=device-width, initial-scale=1"><style>body{background:#eee;width:60vw;margin:15vh auto;font-family:system-ui,sans-serif}h1{font-size:1.5em}div{opacity:0.8}a:link,a:visited{color:#348}</style></head><body><div><h1>Example Domain</h1><p>This domain is for use in documentation examples without needing permission. Avoid use in operations.</p><p><a href="https://iana.o

=== First H1 tags ===
Example Domain

=== First H2 tags ===

=== First P tags ===
This domain is for use in documentation examples without needing permission. Avoid use in operations.
Learn more

=== Tag frequency summary ===
p: 2
html: 1
head: 1
title: 1
link: 1
meta: 1
style: 1
body: 1
div: 1
h1: 1

=== External Links (first 5) ===
https://iana.org/domains/example

=== Meta tags foun

## Optional Example — Parsing a Messy News Page into Useful Data

Goal: Show how unstructured, noisy HTML can still be transformed into repeatable, useful data.

In [17]:
URL_NEWS = "https://www.bbc.com/news"
news_html_path = DATA_RAW / f"bbc_news_{safe_stamp()}.html"

resp = requests.get(URL_NEWS, headers={"User-Agent": "AFE-Course-Notebook/1.0"}, timeout=30)
resp.raise_for_status()
with open(news_html_path, "w", encoding="utf-8") as f:
    f.write(resp.text)
print(f"Downloaded and saved raw HTML to: {news_html_path}")

with open(news_html_path, "r", encoding="utf-8") as f:
    news_html = f.read()
print("\n=== First 500 chars of messy HTML ===")
print(news_html[:500])

soup_news = BeautifulSoup(news_html, "html.parser")

tag_counts_news = collections.Counter([tag.name for tag in soup_news.find_all(True)])
print("\n=== Top 10 HTML tags in BBC page ===")
for tag, count in tag_counts_news.most_common(10):
    print(f"{tag}: {count}")

headlines_data = []
# BBC moved headlines out of <h3> - they are now <h2>, often with a
# data-testid hook. Try the stable semantic tag first, then the test id.
cards = soup_news.select("h2") or soup_news.select('[data-testid="card-headline"]')
for card in cards:
    a = card.find("a") or card.find_parent("a")
    if a and a.get("href") and card.get_text(strip=True):
        headlines_data.append({
            "headline": card.get_text(strip=True),
            "url": a["href"] if a["href"].startswith("http") else f"https://www.bbc.com{a['href']}"
        })

seen = set()
unique_headlines = []
for item in headlines_data:
    if item["headline"] not in seen:
        seen.add(item["headline"])
        unique_headlines.append(item)
headlines_df = pd.DataFrame(unique_headlines[:10])

print("\n=== Extracted BBC Headlines (Top 10) ===")
print(headlines_df)

headlines_csv = DATA_RAW / f"bbc_headlines_{safe_stamp()}.csv"
headlines_df.to_csv(headlines_csv, index=False)
print(f"Headlines saved to: {headlines_csv}")

Downloaded and saved raw HTML to: data/raw/bbc_news_20260814-113150.html

=== First 500 chars of messy HTML ===
<!DOCTYPE html><html lang="en-GB"><head><meta charSet="utf-8" data-next-head=""/><meta name="viewport" content="width=device-width, initial-scale=1" data-next-head=""/><title data-next-head="">BBC News - Breaking news, video and the latest top stories from the U.S. and around the world</title><meta name="description" content="Visit BBC News for the latest news, breaking news, video, audio and analysis. BBC News provides trusted World, U.S. and U.K. news as well as local and regional perspectives

=== Top 10 HTML tags in BBC page ===
div: 674
a: 247
li: 137
span: 93
ul: 88
script: 56
h2: 54
img: 42
path: 41
svg: 34

=== Extracted BBC Headlines (Top 10) ===
                                            headline  \
0  Luigi Mangione intends to plead guilty to fede...   
1  Afghan women answer your questions about life ...   
2  Nigel Farage faces renewed watchdog probe over...   
